# Electroweak Spontaneous Symmetry Breaking Walkthrough

This notebook walks through the broken electroweak sector implemented in the symbolic QFT framework. It uses the public SSB helpers to build the Higgs doublet, expand around the vev, rotate into the physical gauge basis, inspect mass generation, and extract representative tree-level vertices.

The goal is not to hide the physics in validation helpers. Each section prints or extracts the explicit symbolic structures that the framework stores in the broken-phase `Model`.

## 1. Imports and Notebook Setup

The path setup lets the notebook run from either the repository root or the `notebooks/` directory.

In [14]:
from __future__ import annotations

import sys
from fractions import Fraction
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from symbolica import Expression, S

from model import (
    CKMChargedCurrentAssignment,
    DiagonalYukawaAssignment,
    ElectroweakGaugeFixing,
    FLAVOR_INDEX,
    Field,
    FlavorMatrix,
    FlavorMatrixYukawaAssignment,
    SPINOR_INDEX,
    build_broken_electroweak_sector,
    electroweak_e,
    electroweak_mw,
    electroweak_mz,
    standard_model_higgs_doublet,
)
from symbolic.vertex_engine import Delta, I, bis, pcomp, pi
from symbolic.spenso_structures import lorentz_metric

HALF = Expression.num(1) / Expression.num(2)
SQRT_HALF = HALF ** HALF

def canon(expr):
    return expr.expand().to_canonical_string()

def has_local_two_point_mass_term(lagrangian, field):
    for term in lagrangian.terms:
        if term.derivatives:
            continue
        term_fields = tuple((occ.field, occ.conjugated) for occ in term.fields)
        if term_fields == ((field, False), (field, False)):
            return True
    return False

## 2. Symbols and Matter Fields

We keep all parameters symbolic. The diagonal electron Yukawa demonstrates the original minimal Dirac mass path. The flavored up/down fields demonstrate open flavor indices and CKM-like charged currents.

In [15]:
g1, g2, v = S("g1", "g2", "v")
ye, lamH = S("y_e", "lamH")
xiW, xiZ, xiA = S("xiW", "xiZ", "xiA")
d = S("d")
q1, q2, q3, q4 = S("q1", "q2", "q3", "q4")
D2 = (2 * pi) ** d * Delta(q1 + q2)
D3 = (2 * pi) ** d * Delta(q1 + q2 + q3)
D4 = (2 * pi) ** d * Delta(q1 + q2 + q3 + q4)

electron = Field(
    "e",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("e0"),
    conjugate_symbol=S("ebar0"),
    indices=(SPINOR_INDEX,),
)
up_flavored = Field(
    "u",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("u0"),
    conjugate_symbol=S("ubar0"),
    indices=(SPINOR_INDEX, FLAVOR_INDEX),
)
down_flavored = Field(
    "d",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("d0"),
    conjugate_symbol=S("dbar0"),
    indices=(SPINOR_INDEX, FLAVOR_INDEX),
)


Yukawa matrices & CKM matrix
not yet in spenso...


In [16]:

Yu = FlavorMatrix("Yu")
VCKM = FlavorMatrix("VCKM")

## 3. Build the Broken Electroweak Sector

`build_broken_electroweak_sector` returns an inspectable bundle: source fields, physical fields, mixing relations, masses, optional Higgs potential data, optional gauge fixing data, and the compiled `Model` used for Feynman-rule extraction.

In [30]:
# --- 1. Define the Higgs field (SU(2) doublet with Y = 1/2) ---
# This is the standard SM Higgs; you can override it if needed.
higgs_doublet = standard_model_higgs_doublet()


# --- 2. Build the broken electroweak sector ---
# This is the central helper:
# it constructs the Lagrangian AFTER spontaneous symmetry breaking (SSB)
# and returns a bundle with:
#   - physical fields (h, G0, W±, Z, A, ghosts, fermions)
#   - mixing relations (W±, Z/A)
#   - masses (MW, MZ, Mh, fermions)
#   - optional gauge + Higgs + ghost sectors
#   - a compiled Model for Feynman rules

broken = build_broken_electroweak_sector(
    # --- gauge couplings ---
    g1=g1,   # U(1)_Y coupling
    g2=g2,   # SU(2)_L coupling

    # --- Higgs vev ---
    vev=v,   # sets the electroweak scale and all masses

    # --- include full gauge sector in physical basis ---
    include_gauge_sector=True,

    # --- Higgs potential (controls Higgs mass and self-interactions) ---
    higgs_quartic=lamH,

    # --- R_xi gauge fixing (introduces gauge parameters + ghosts) ---
    gauge_fixing=ElectroweakGaugeFixing(
        xi_w=xiW,
        xi_z=xiZ,
        xi_a=xiA,
    ),

    # --- input Higgs field ---
    higgs_doublet=higgs_doublet,

    # --- diagonal Yukawa: simple mass term example (electron) ---
    yukawas=(
        DiagonalYukawaAssignment(
            electron,
            ye,
            label="electron Yukawa",
        ),
    ),

    # --- matrix Yukawa: flavor-dependent masses (e.g. up quarks) ---
    matrix_yukawas=(
        FlavorMatrixYukawaAssignment(
            up_flavored,
            Yu,   # Yukawa matrix Y_u(i,j)
            label="up matrix Yukawa",
        ),
    ),

    # --- charged-current mixing (CKM matrix) ---
    # generates W+ u_i -> d_j interactions with V_CKM(i,j)
    charged_current_mixings=(
        CKMChargedCurrentAssignment(
            up_flavored,
            down_flavored,
            VCKM,
            label="CKM current",
        ),
    ),
)


# --- 3. Extract the Lagrangian ---
# This is the compiled object used to compute Feynman rules
L = broken.model.lagrangian()


# --- 4. Quick inspection ---
print(broken.model.name)

# Physical (mass eigenstate) fields after SSB
print("physical fields:", [field.name for field in broken.model.fields])

# Number of local interaction terms in the Lagrangian
print("number of local terms:", len(L.terms))

print("MW =", broken.masses.mw)
print("MZ =", broken.masses.mz)
print("Mh =", broken.masses.mh)

EW-broken-phase
physical fields: ['h', 'G0', 'Gp', 'Wp', 'Z', 'A', 'ghWp', 'ghZ', 'ghA', 'e', 'u', 'd']
number of local terms: 78
MW = 1/2*g2*v
MZ = 1/2*v*(g1^2+g2^2)^(1/2)
Mh = (2*v^2*lamH)^(1/2)


## 4. Higgs Doublet and VEV Expansion

The unbroken source Higgs is a complex SU(2) fundamental scalar with hypercharge `Y = 1/2`. The broken expansion is stored as human-readable linear relations rather than hidden substitutions.

In [18]:
print("Higgs doublet:")
print(higgs_doublet)
print("hypercharge Y =", higgs_doublet.quantum_numbers["Y"])

print("\nExpansion around the vev:")
for relation in broken.higgs_expansion:
    print(" ", relation)

Higgs doublet:
Field(name='H', spin=0, self_conjugate=False, indices=(IndexType(name='WeakFund', representation=Representation { rep: Dualizable(3), dim: Concrete(2) }, kind='weak_fund', prefix='w'),), kind='scalar', statistics='boson', symbol=H0, conjugate_symbol=Hdag0, mass=None, quantum_numbers={'Y': 1/2})
hypercharge Y = 1/2

Expansion around the vev:
  H[1] = 1 * Gp
  H[2] = (1/2)^(1/2)*v + (1/2)^(1/2) * h + 1𝑖*(1/2)^(1/2) * G0
  Hdag[1] = 1 * Gp.bar
  Hdag[2] = (1/2)^(1/2)*v + (1/2)^(1/2) * h + -1𝑖*(1/2)^(1/2) * G0


## 5. Electroweak Gauge Rotations

The charged and neutral rotations are explicit metadata. This keeps the physical basis auditable: `W1, W2 -> W+, W-` and `W3, B -> Z, A`.

In [19]:
print("Charged rotation:")
for relation in broken.charged_mixing:
    print(" ", relation)

print("\nNeutral rotation:")
for relation in broken.neutral_mixing:
    print(" ", relation)

print("\nPhysical gauge couplings:")
print(" e     =", broken.gauge_couplings.electric_charge)
print(" gWWA  =", broken.gauge_couplings.g_ww_a)
print(" gWWZ  =", broken.gauge_couplings.g_ww_z)

Charged rotation:
  W+ = (1/2)^(1/2) * W1 + -1𝑖*(1/2)^(1/2) * W2
  W- = (1/2)^(1/2) * W1 + 1𝑖*(1/2)^(1/2) * W2

Neutral rotation:
  Z = g2/(g1^2+g2^2)^(1/2) * W3 + -g1/(g1^2+g2^2)^(1/2) * B
  A = g1/(g1^2+g2^2)^(1/2) * W3 + g2/(g1^2+g2^2)^(1/2) * B

Physical gauge couplings:
 e     = g1*g2/(g1^2+g2^2)^(1/2)
 gWWA  = g1*g2/(g1^2+g2^2)^(1/2)
 gWWZ  = g2^2/(g1^2+g2^2)^(1/2)


## 6. Mass Generation from the Higgs Kinetic Term

The broken Higgs kinetic term generates `M_W`, `M_Z`, no photon mass, and the expected `hVV` interactions. The photon check below is deliberately structural: there is no local `A A` mass term in the broken Higgs sector.

In [20]:
wp = broken.fields.charged_w
z = broken.fields.z_boson
photon = broken.fields.photon
h = broken.fields.higgs

print("MW =", broken.masses.mw)
print("MZ =", broken.masses.mz)
print("photon mass =", broken.masses.photon)
print("local photon mass term present?", has_local_two_point_mass_term(L, photon))

print("\nΓ(W-, W+):")
print(L.feynman_rule(wp.bar, wp))

print("\nΓ(Z, Z):")
print(L.feynman_rule(z, z))

print("\nΓ(h, W-, W+):")
print(L.feynman_rule(h, wp.bar, wp))

print("\nΓ(h, Z, Z):")
print(L.feynman_rule(h, z, z))

MW = 1/2*g2*v
MZ = 1/2*v*(g1^2+g2^2)^(1/2)
photon mass = 0
local photon mass term present? False

Γ(W-, W+):
1𝑖/4*g2^2*v^2*(2*𝜋)^d*g(mink(4, i1),mink(4, i2))*Delta(q1+q2)+1𝑖*(2*𝜋)^d*pcomp(q1,i1)*pcomp(q2,i2)*Delta(q1+q2)/xiW-1𝑖*(2*𝜋)^d*pcomp(q1,i2)*pcomp(q2,i1)*Delta(q1+q2)+1𝑖*(2*𝜋)^d*g(mink(4, i1),mink(4, i2))*pcomp(q1,rho_Wp_kin)*pcomp(q2,rho_Wp_kin)*Delta(q1+q2)

Γ(Z, Z):
1𝑖/4*g1^2*v^2*(2*𝜋)^d*g(mink(4, i1),mink(4, i2))*Delta(q1+q2)+1𝑖/4*g2^2*v^2*(2*𝜋)^d*g(mink(4, i1),mink(4, i2))*Delta(q1+q2)+1𝑖*(2*𝜋)^d*pcomp(q1,i1)*pcomp(q2,i2)*Delta(q1+q2)/xiZ-1𝑖*(2*𝜋)^d*pcomp(q1,i2)*pcomp(q2,i1)*Delta(q1+q2)+1𝑖*(2*𝜋)^d*g(mink(4, i1),mink(4, i2))*pcomp(q1,rho_Z_kin)*pcomp(q2,rho_Z_kin)*Delta(q1+q2)

Γ(h, W-, W+):
1𝑖/2*g2^2*v*(2*𝜋)^d*g(mink(4, i1),mink(4, i2))*Delta(q1+q2+q3)

Γ(h, Z, Z):
1𝑖/2*g1^2*v*(2*𝜋)^d*g(mink(4, i1),mink(4, i2))*Delta(q1+q2+q3)+1𝑖/2*g2^2*v*(2*𝜋)^d*g(mink(4, i1),mink(4, i2))*Delta(q1+q2+q3)


A compact symbolic check of the mass terms is still possible. The expected forms are kept close to the physics: `i M^2 g_{mu nu}` for the vector two-point mass contribution.

### Minimal electroweak SSB mass check

This cell verifies that the **minimal broken electroweak sector** (without gauge kinetic terms or gauge fixing) produces the correct pure mass vertices for the gauge bosons:

$$
M_W^2 = \frac{g_2^2 v^2}{4}, \qquad
M_Z^2 = \frac{(g_1^2 + g_2^2) v^2}{4}.
$$

By comparing the computed two-point Feynman rules with the expected expressions $ i M^2 g_{\mu\nu} $, we isolate and confirm that mass generation from the Higgs mechanism is implemented correctly.

In [31]:
# --- expected mass-squared values from the Higgs mechanism ---
mw_sq = electroweak_mw(g2, v) ** 2
mz_sq = electroweak_mz(g1, g2, v) ** 2

# --- build the *minimal* broken EW sector ---
# (no gauge kinetic terms, no gauge fixing, no Yukawas)
# → isolates pure Higgs-induced mass terms
broken_minimal = build_broken_electroweak_sector(
    g1=g1,
    g2=g2,
    vev=v,
    higgs_doublet=standard_model_higgs_doublet(),
)

# compiled Lagrangian
L_minimal = broken_minimal.model.lagrangian()

# physical fields
wp_min = broken_minimal.fields.charged_w
z_min = broken_minimal.fields.z_boson

# --- expected two-point vertices: i M^2 g_{μν} ---
expected_w_mass = I * mw_sq * lorentz_metric(S("i1"), S("i2")) * D2
expected_z_mass = I * mz_sq * lorentz_metric(S("i1"), S("i2")) * D2

# --- compare computed vs expected ---
print("W mass check:", canon(L_minimal.feynman_rule(wp_min.bar, wp_min)) == canon(expected_w_mass))
print("Z mass check:", canon(L_minimal.feynman_rule(z_min, z_min)) == canon(expected_z_mass))

W mass check: True
Z mass check: True


## 7. Physical Gauge Self-Interactions

With `include_gauge_sector=True`, the broken electroweak model includes the physical-basis gauge kinetic and self-interaction sector.

The vertices below show the expected electroweak Yang–Mills structures after rotating from the gauge basis \((W^1, W^2, W^3, B)\) to the physical basis \((W^\pm, Z, A)\):

- \(WWA\), controlled by the electric charge \(e\)
- \(WWZ\), controlled by \(g_{WWZ}\)
- quartic vertices such as \(WWAZ\) and \(WWAA\)

In [36]:
print("Physical couplings:")
print(" e    =", broken.gauge_couplings.electric_charge)
print(" gWWA =", broken.gauge_couplings.g_ww_a)
print(" gWWZ =", broken.gauge_couplings.g_ww_z)
print()

print("Γ(W-, W+, A):")
print(L.feynman_rule(wp.bar, wp, photon))

print("\nΓ(W-, W+, Z):")
print(L.feynman_rule(wp.bar, wp, z))

print("\nΓ(W-, W+, A, Z):")
print(L.feynman_rule(wp.bar, wp, photon, z))

print("\nΓ(W-, W+, A, A):")
print(L.feynman_rule(wp.bar, wp, photon, photon))

Physical couplings:
 e    = g1*g2/(g1^2+g2^2)^(1/2)
 gWWA = g1*g2/(g1^2+g2^2)^(1/2)
 gWWZ = g2^2/(g1^2+g2^2)^(1/2)

Γ(W-, W+, A):
1𝑖*g1*g2*(2*𝜋)^d*g(mink(4, i1),mink(4, i2))*pcomp(q1,i3)*Delta(q1+q2+q3)/(g1^2+g2^2)^(1/2)-1𝑖*g1*g2*(2*𝜋)^d*g(mink(4, i1),mink(4, i2))*pcomp(q2,i3)*Delta(q1+q2+q3)/(g1^2+g2^2)^(1/2)-1𝑖*g1*g2*(2*𝜋)^d*g(mink(4, i1),mink(4, i3))*pcomp(q1,i2)*Delta(q1+q2+q3)/(g1^2+g2^2)^(1/2)+1𝑖*g1*g2*(2*𝜋)^d*g(mink(4, i1),mink(4, i3))*pcomp(q3,i2)*Delta(q1+q2+q3)/(g1^2+g2^2)^(1/2)+1𝑖*g1*g2*(2*𝜋)^d*g(mink(4, i2),mink(4, i3))*pcomp(q2,i1)*Delta(q1+q2+q3)/(g1^2+g2^2)^(1/2)-1𝑖*g1*g2*(2*𝜋)^d*g(mink(4, i2),mink(4, i3))*pcomp(q3,i1)*Delta(q1+q2+q3)/(g1^2+g2^2)^(1/2)

Γ(W-, W+, Z):
1𝑖*g2^2*(2*𝜋)^d*g(mink(4, i1),mink(4, i2))*pcomp(q1,i3)*Delta(q1+q2+q3)/(g1^2+g2^2)^(1/2)-1𝑖*g2^2*(2*𝜋)^d*g(mink(4, i1),mink(4, i2))*pcomp(q2,i3)*Delta(q1+q2+q3)/(g1^2+g2^2)^(1/2)-1𝑖*g2^2*(2*𝜋)^d*g(mink(4, i1),mink(4, i3))*pcomp(q1,i2)*Delta(q1+q2+q3)/(g1^2+g2^2)^(1/2)+1𝑖*g2^2*(2*𝜋)^d*g(mink(4, i1),mink(4, i

## 8. Higgs Potential: Mass and Self-Couplings

Passing `higgs_quartic=lamH` adds the broken Higgs potential.  
The derived metadata exposes:

- $m_h^2 = 2\lambda_H v^2$
- $m_h = \sqrt{2\lambda_H v^2}$
- the broken-phase $hhh$ and $hhhh$ coupling coefficients

The Feynman rules below confirm the corresponding Higgs two-, three-, and four-point vertices.

In [37]:
print("mh^2 =", broken.higgs_potential.mh_sq)
print("mh   =", broken.higgs_potential.mh)
print("hhh Coupling coefficient  =", broken.higgs_potential.hhh_vertex_coefficient)
print("hhhh Coupling coefficient =", broken.higgs_potential.hhhh_vertex_coefficient)

print("\nΓ(h, h):")
print(L.feynman_rule(h, h))

print("\nΓ(h, h, h):")
print(L.feynman_rule(h, h, h))

print("\nΓ(h, h, h, h):")
print(L.feynman_rule(h, h, h, h))

mh^2 = 2*v^2*lamH
mh   = (2*v^2*lamH)^(1/2)
hhh Coupling coefficient  = -6*v*lamH
hhhh Coupling coefficient = -6*lamH

Γ(h, h):
-2𝑖*v^2*lamH*(2*𝜋)^d*Delta(q1+q2)-1𝑖*(2*𝜋)^d*pcomp(q1,mu)*pcomp(q2,mu)*Delta(q1+q2)

Γ(h, h, h):
-6𝑖*v*lamH*(2*𝜋)^d*Delta(q1+q2+q3)

Γ(h, h, h, h):
-6𝑖*lamH*(2*𝜋)^d*Delta(q1+q2+q3+q4)


## 9. R_xi Gauge Fixing and Goldstones

The broken sector can include physical-basis \($R_\xi$\) gauge fixing.

These terms:

- cancel the unphysical mixing between Goldstones and gauge bosons
  $$
  \Gamma(G^0, Z) = 0, \quad \Gamma(G^\pm, W^\mp) = 0
    $$
- give the Goldstones gauge-dependent masses
    $$
  m_{G^0}^2 = \xi_Z M_Z^2, \quad m_{G^\pm}^2 = \xi_W M_W^2
    $$

In [24]:
g0 = broken.fields.goldstone_neutral
gp = broken.fields.goldstone_charged

print("Γ(G0, Z) after gauge fixing:")
print(L.feynman_rule(g0, z))

print("\nΓ(G+, W-) after gauge fixing:")
print(L.feynman_rule(gp, wp.bar))

print("\nΓ(G0, G0):")
print(L.feynman_rule(g0, g0))

print("\nΓ(G-, G+):")
print(L.feynman_rule(gp.bar, gp))

Γ(G0, Z) after gauge fixing:
0

Γ(G+, W-) after gauge fixing:
0

Γ(G0, G0):
-1𝑖/4*g1^2*v^2*xiZ*(2*𝜋)^d*Delta(q1+q2)-1𝑖/4*g2^2*v^2*xiZ*(2*𝜋)^d*Delta(q1+q2)-1𝑖*(2*𝜋)^d*pcomp(q1,mu)*pcomp(q2,mu)*Delta(q1+q2)

Γ(G-, G+):
-1𝑖/4*g2^2*v^2*xiW*(2*𝜋)^d*Delta(q1+q2)-1𝑖*(2*𝜋)^d*pcomp(q1,mu)*pcomp(q2,mu)*Delta(q1+q2)


The vanishing mixed vertices confirm that gauge fixing removes the unphysical
Goldstone–vector mixing. The remaining two-point functions show that the
Goldstones acquire masses proportional to the gauge-fixing parameters,
as expected in \(R_\xi\) gauges.

## 10. Broken-Phase Ghost Sector

In \(R_\xi\) gauges, each gauge boson is accompanied by a Faddeev–Popov ghost field.

These are unphysical scalar fields that:
- cancel gauge-dependent contributions in loop diagrams
- ensure unitarity and gauge invariance of the theory

Here we inspect:
- ghost propagators (bilinears)
- ghost–gauge interactions
- ghost–Goldstone interactions induced by gauge fixing

In [39]:
c_w = broken.fields.ghost_charged
c_z = broken.fields.ghost_z
c_a = broken.fields.ghost_photon

print("Ghost propagators:")
print("Γ(cWbar, cW):")
print(L.feynman_rule(c_w.bar, c_w))

print("\nΓ(cZbar, cZ):")
print(L.feynman_rule(c_z.bar, c_z))

print("\nΓ(cAbar, cA):")
print(L.feynman_rule(c_a.bar, c_a))

print("\nGhost–gauge vertices:")
print("Γ(cWbar, A, cW):")
print(L.feynman_rule(c_w.bar, photon, c_w))

print("\nΓ(cWbar, W+, cA):")
print(L.feynman_rule(c_w.bar, wp, c_a))

print("\nGhost–Goldstone vertices:")
print("Γ(cZbar, cW, G-):")
print(L.feynman_rule(c_z.bar, c_w, gp.bar))

Ghost propagators:
Γ(cWbar, cW):
-1𝑖/4*g2^2*v^2*xiW*(2*𝜋)^d*Delta(q1+q2)-1𝑖*(2*𝜋)^d*pcomp(q1,mu_ghWp_ghost)*pcomp(q2,mu_ghWp_ghost)*Delta(q1+q2)

Γ(cZbar, cZ):
-1𝑖/4*g1^2*v^2*xiZ*(2*𝜋)^d*Delta(q1+q2)-1𝑖/4*g2^2*v^2*xiZ*(2*𝜋)^d*Delta(q1+q2)-1𝑖*(2*𝜋)^d*pcomp(q1,mu_ghZ_ghost)*pcomp(q2,mu_ghZ_ghost)*Delta(q1+q2)

Γ(cAbar, cA):
-1𝑖*(2*𝜋)^d*pcomp(q1,mu_ghA_ghost)*pcomp(q2,mu_ghA_ghost)*Delta(q1+q2)

Ghost–gauge vertices:
Γ(cWbar, A, cW):
-1𝑖*g1*g2*(2*𝜋)^d*pcomp(q1,i1)*Delta(q1+q2+q3)/(g1^2+g2^2)^(1/2)

Γ(cWbar, W+, cA):
1𝑖*g1*g2*(2*𝜋)^d*pcomp(q1,i1)*Delta(q1+q2+q3)/(g1^2+g2^2)^(1/2)

Ghost–Goldstone vertices:
Γ(cZbar, cW, G-):
1𝑖/4*g2*v*xiZ*(2*𝜋)^d*(g1^2+g2^2)^(1/2)*Delta(q1+q2+q3)


## 11. Yukawa Masses and Higgs-Fermion Vertices

The diagonal Yukawa path produces a scalar Dirac mass and `h fbar f`. The matrix path keeps open flavor labels, so the mass and Higgs vertices carry `Yu(i,j)`.

In [26]:
for fermion, mass in broken.masses.fermions:
    print(f"M[{fermion.name}] =", mass)

print("\nDiagonal electron Yukawa Γ(ebar, e):")
print(L.feynman_rule(electron.bar, electron))

print("\nDiagonal electron Yukawa Γ(ebar, e, h):")
print(L.feynman_rule(electron.bar, electron, h))

print("\nMatrix Yukawa Γ(ubar, u):")
print(L.feynman_rule(up_flavored.bar, up_flavored))

print("\nMatrix Yukawa Γ(ubar, u, h):")
print(L.feynman_rule(up_flavored.bar, up_flavored, h))

M[e] = (1/2)^(1/2)*v*y_e
M[u] = (1/2)^(1/2)*v*Yu

Diagonal electron Yukawa Γ(ebar, e):
-1𝑖*(1/2)^(1/2)*v*y_e*(2*𝜋)^d*g(bis(4, i1),bis(4, i2))*Delta(q1+q2)

Diagonal electron Yukawa Γ(ebar, e, h):
-1𝑖*(1/2)^(1/2)*y_e*(2*𝜋)^d*g(bis(4, i1),bis(4, i2))*Delta(q1+q2+q3)

Matrix Yukawa Γ(ubar, u):
-1𝑖*(1/2)^(1/2)*v*(2*𝜋)^d*g(bis(4, i1),bis(4, i3))*Delta(q1+q2)*Yu(i2,i4)

Matrix Yukawa Γ(ubar, u, h):
-1𝑖*(1/2)^(1/2)*(2*𝜋)^d*g(bis(4, i1),bis(4, i3))*Delta(q1+q2+q3)*Yu(i2,i4)


## 12. CKM-Like Charged Currents

The charged-current metadata adds left-chiral `W+` and `W-` interactions. The backward vertex uses the dagger structure by reversing the symbolic flavor-matrix indices.

In [27]:
print("Γ(ubar, d, W+):")
print(L.feynman_rule(up_flavored.bar, down_flavored, wp))

print("\nΓ(dbar, u, W-):")
print(L.feynman_rule(down_flavored.bar, up_flavored, wp.bar))

Γ(ubar, d, W+):
1𝑖/2*(1/2)^(1/2)*g2*(2*𝜋)^d*gamma(bis(4, i1),bis(4, alpha_u_d_cc_mid),mink(4, i5))*gamma5(bis(4, alpha_u_d_cc_mid),bis(4, i3))*Delta(q1+q2+q3)*VCKM(i2,i4)-1𝑖/2*(1/2)^(1/2)*g2*(2*𝜋)^d*g(bis(4, alpha_u_d_cc_mid),bis(4, i3))*gamma(bis(4, i1),bis(4, alpha_u_d_cc_mid),mink(4, i5))*Delta(q1+q2+q3)*VCKM(i2,i4)

Γ(dbar, u, W-):
1𝑖/2*(1/2)^(1/2)*g2*(2*𝜋)^d*gamma(bis(4, i1),bis(4, alpha_d_u_cc_mid),mink(4, i5))*gamma5(bis(4, alpha_d_u_cc_mid),bis(4, i3))*Delta(q1+q2+q3)*VCKM(i4,i2)-1𝑖/2*(1/2)^(1/2)*g2*(2*𝜋)^d*g(bis(4, alpha_d_u_cc_mid),bis(4, i3))*gamma(bis(4, i1),bis(4, alpha_d_u_cc_mid),mink(4, i5))*Delta(q1+q2+q3)*VCKM(i4,i2)


## 13. What This Walkthrough Covers

- Standard complex Higgs doublet with `Y = 1/2`.
- Explicit vev expansion into `h`, `G0`, and `G+`.
- Charged and neutral physical gauge rotations.
- `W` and `Z` mass generation with a massless photon.
- `hWW`, `hZZ`, gauge self-interactions, Higgs potential self-couplings.
- `R_xi` cancellation of Goldstone-vector mixing and physical-basis ghosts.
- Diagonal and matrix Yukawa masses plus CKM-like charged currents.

Remaining future steps are full flavor diagonalization, deriving CKM from Yukawa rotations, a complete BRST-level ghost organization, and assembling the full broken Standard Model sector by sector.